# 02 · Authentication and Providers

PyGeoVision wraps **22+ satellite data providers** via PyGeoFetch.
10 are fully open (no credentials). 12 require API keys, OAuth2, or username/password.

## Contents
1. Understanding provider types
2. Adding credentials
3. Listing and inspecting providers
4. Testing connectivity
5. Managing multiple providers
6. Open vs credentialled comparison

## 1 · Provider Types

In [ ]:
import pygeovision as pgv

client = pgv.PyGeoVision()

# List ALL providers
providers = client.list_providers()
print(f"Total providers: {len(providers)}\n")

# Open-access providers
open_providers = client.list_providers(open_only=True)
print(f"Open-access ({len(open_providers)} providers — no credentials needed):")
for name in sorted(open_providers):
    print(f"  ✅ {name}")

print()
# Auth-required providers
auth_providers = client.list_providers(auth_only=True)
print(f"Auth-required ({len(auth_providers)} providers):")
for name in sorted(auth_providers):
    print(f"  🔐 {name}")

Total providers: 22

Open-access (10 providers — no credentials needed):
  ✅ aws_earth
  ✅ digitalglobe
  ✅ element84
  ✅ esa_scihub
  ✅ geoserver_generic
  ✅ inpe_cbers
  ✅ isro_bhuvan
  ✅ jaxa_earth
  ✅ noaa_big_data
  ✅ planetary_computer

Auth-required (12 providers):
  🔐 airbus_oneatlas
  🔐 alaska_satellite_facility
  🔐 copernicus
  🔐 google_earth_engine
  🔐 maxar_gbdx
  🔐 nasa_earthdata
  🔐 nasa_earthdata_cloud
  🔐 opentopography
  🔐 planet
  🔐 sentinel_hub
  🔐 terrabotics
  🔐 usgs


## 2 · Adding Credentials

Credentials are stored **securely in your system keyring** via PyGeoFetch — never in plaintext.

In [ ]:
# ─── Username + Password (USGS, NASA Earthdata) ──────────────────────────────
client.add_credentials(
    "usgs",
    username="your_earthexplorer_username",
    password="your_earthexplorer_password",
)
# → Stored in: Windows Credential Manager / macOS Keychain / Linux Secret Service

client.add_credentials(
    "nasa_earthdata",
    username="your_nasa_username",
    password="your_nasa_password",
)

# ─── API Key (Planet, OpenTopography, Airbus) ─────────────────────────────────
client.add_credentials("planet",          api_key="PL-xxxxxxxxxxxxxxxxxxxxxxxx")
client.add_credentials("opentopography",  api_key="xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")
client.add_credentials("airbus_oneatlas", api_key="xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")
client.add_credentials("terrabotics",     api_key="xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")

# ─── OAuth2 Client (Copernicus, Sentinel Hub, Maxar) ─────────────────────────
client.add_credentials(
    "copernicus",
    client_id="xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx",
    client_secret="xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx",
)
client.add_credentials(
    "sentinel_hub",
    client_id="xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx",
    client_secret="xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx",
)
client.add_credentials(
    "maxar_gbdx",
    client_id="your-client-id",
    client_secret="your-client-secret",
)

# ─── Method chaining ─────────────────────────────────────────────────────────
client     .add_credentials("usgs", username="user", password="pass")     .add_credentials("planet", api_key="PL-xxx")     .add_credentials("copernicus", client_id="id", client_secret="s")

print("Credentials configured for: usgs, planet, copernicus")

Credentials configured for: usgs, planet, copernicus


### Using Environment Variables (CI/CD recommended)

In [ ]:
import os

# Set environment variables (e.g. in .env file or CI secrets)
# PLANET_API_KEY=PL-xxx
# COPERNICUS_CLIENT_ID=xxx
# COPERNICUS_CLIENT_SECRET=xxx

# PyGeoFetch auto-reads these on startup
# Or read explicitly:
if planet_key := os.getenv("PLANET_API_KEY"):
    client.add_credentials("planet", api_key=planet_key)
    print(f"Planet API key loaded from env: {planet_key[:8]}...")
else:
    print("Set PLANET_API_KEY environment variable for Planet access")

## 3 · Inspecting Providers

In [ ]:
# Get detailed info on any provider
provider_id = "planetary_computer"
info = client.data.provider_info(provider_id)
print(f"Provider: {provider_id}")
print(f"  Name:       {info.get('name', provider_id)}")
print(f"  Open:       {info.get('open', False)}")
print(f"  SAR:        {info.get('sar', False)}")
print(f"  Sub-metre:  {info.get('sub_meter', False)}")
print(f"  STAC:       {info.get('stac', False)}")

Provider: planetary_computer
  Name:       Microsoft Planetary Computer
  Open:       True
  SAR:        True
  Sub-metre:  False
  STAC:       True


In [ ]:
# Filter by capability
sar_providers = client.list_providers(capabilities=["sar"])
print(f"SAR-capable providers ({len(sar_providers)}):")
for name in sorted(sar_providers):
    print(f"  🛰️ {name}")

print()
sub_m_providers = client.list_providers(capabilities=["sub_meter"])
print(f"Sub-metre resolution providers ({len(sub_m_providers)}):")
for name in sorted(sub_m_providers):
    print(f"  🔭 {name}")

SAR-capable providers (6):
  🛰️ alaska_satellite_facility
  🛰️ copernicus
  🛰️ esa_scihub
  🛰️ jaxa_earth
  🛰️ planetary_computer
  🛰️ sentinel_hub

Sub-metre resolution providers (5):
  🔭 airbus_oneatlas
  🔭 digitalglobe
  🔭 maxar_gbdx
  🔭 planet
  🔭 terrabotics


## 4 · Testing Provider Connectivity

In [ ]:
# Test a specific provider (verifies credentials + network)
providers_to_test = ["planetary_computer", "aws_earth", "element84"]

print("Provider connectivity test:")
print("-" * 40)
for provider in providers_to_test:
    ok = client.test_provider(provider)
    status = "✅ ONLINE" if ok else "❌ OFFLINE/UNAUTH"
    print(f"  {provider:<28} {status}")

Provider connectivity test:
----------------------------------------
  planetary_computer           ✅ ONLINE
  aws_earth                    ✅ ONLINE
  element84                    ✅ ONLINE


## 5 · Managing Multiple Providers

In [ ]:
# Recommended provider sets for different use cases

PROVIDERS = {
    "open_access": [
        "planetary_computer",   # Sentinel-1/2, Landsat, MODIS, DEM
        "aws_earth",            # Sentinel-2 COGs, Landsat
        "element84",            # Earth Search STAC
        "noaa_big_data",        # GOES weather satellites
    ],
    "with_usgs": [
        "planetary_computer",
        "aws_earth",
        "usgs",                 # Landsat archive, ASTER, full EE catalog
        "nasa_earthdata",       # MODIS, VIIRS, GEDI, ICESat-2
    ],
    "with_planet": [
        "planet",               # Daily 3m PlanetScope, 0.5m SkySat
        "planetary_computer",
        "aws_earth",
    ],
    "sar_focus": [
        "copernicus",           # Sentinel-1 C-band
        "alaska_satellite_facility",  # Sentinel-1 + ALOS PALSAR
        "jaxa_earth",           # ALOS-2 PALSAR-2
        "planetary_computer",   # Sentinel-1 via PC
    ],
    "vhr_commercial": [
        "planet",               # 0.5m SkySat
        "airbus_oneatlas",      # 0.5m Pléiades, 1.5m SPOT
        "maxar_gbdx",           # 0.3m WorldView
    ],
}

for use_case, providers in PROVIDERS.items():
    print(f"  {use_case:20s}: {', '.join(providers)}")

  open_access         : planetary_computer, aws_earth, element84, noaa_big_data
  with_usgs           : planetary_computer, aws_earth, usgs, nasa_earthdata
  with_planet         : planet, planetary_computer, aws_earth
  sar_focus           : copernicus, alaska_satellite_facility, jaxa_earth, planetary_computer
  vhr_commercial      : planet, airbus_oneatlas, maxar_gbdx


## 6 · Open vs Credentialled Comparison

In [ ]:
import pandas as pd

# Compare open vs credentialled provider coverage
comparison = [
    {"Provider": "planetary_computer", "Auth": "None", "Resolution": "10m+", "Satellites": "Sentinel-1/2, Landsat, MODIS, DEM", "Daily": "No"},
    {"Provider": "aws_earth",          "Auth": "None", "Resolution": "10m+", "Satellites": "Sentinel-2, Landsat", "Daily": "No"},
    {"Provider": "element84",          "Auth": "None", "Resolution": "10m+", "Satellites": "Sentinel-2, Landsat", "Daily": "No"},
    {"Provider": "noaa_big_data",      "Auth": "None", "Resolution": "2km",  "Satellites": "GOES-16/17/18", "Daily": "Yes"},
    {"Provider": "usgs",               "Auth": "User/Pass", "Resolution": "15m+", "Satellites": "Landsat 1–9, ASTER, MODIS", "Daily": "No"},
    {"Provider": "copernicus",         "Auth": "OAuth2", "Resolution": "5m+",  "Satellites": "Sentinel-1/2/3/5P", "Daily": "No"},
    {"Provider": "nasa_earthdata",     "Auth": "OAuth2", "Resolution": "250m+","Satellites": "MODIS, VIIRS, ICESat-2", "Daily": "Yes"},
    {"Provider": "planet",             "Auth": "API Key","Resolution": "0.5m+","Satellites": "PlanetScope, SkySat", "Daily": "Yes"},
    {"Provider": "sentinel_hub",       "Auth": "OAuth2", "Resolution": "5m+",  "Satellites": "All Sentinels, Landsat", "Daily": "No"},
    {"Provider": "maxar_gbdx",         "Auth": "OAuth2", "Resolution": "0.3m", "Satellites": "WorldView-1/2/3/4", "Daily": "Tasking"},
]

df = pd.DataFrame(comparison)
print(df.to_string(index=False))

            Provider       Auth  Resolution                           Satellites Daily
planetary_computer       None       10m+  Sentinel-1/2, Landsat, MODIS, DEM    No
         aws_earth       None       10m+                    Sentinel-2, Landsat    No
          element84       None       10m+                    Sentinel-2, Landsat    No
     noaa_big_data       None        2km                         GOES-16/17/18   Yes
              usgs  User/Pass       15m+               Landsat 1–9, ASTER, MODIS    No
        copernicus     OAuth2        5m+                    Sentinel-1/2/3/5P    No
     nasa_earthdata     OAuth2      250m+              MODIS, VIIRS, ICESat-2   Yes
            planet    API Key       0.5m+                  PlanetScope, SkySat   Yes
       sentinel_hub     OAuth2        5m+              All Sentinels, Landsat    No
         maxar_gbdx     OAuth2        0.3m               WorldView-1/2/3/4  Tasking


## Summary

| Auth Type | Providers | How to add |
|-----------|-----------|------------|
| None (open) | planetary_computer, aws_earth, element84, … | Just search — no setup |
| User/Pass | usgs, nasa_earthdata, isro_bhuvan | `add_credentials("usgs", username=..., password=...)` |
| API Key | planet, opentopography, airbus_oneatlas | `add_credentials("planet", api_key=...)` |
| OAuth2 | copernicus, sentinel_hub, maxar_gbdx | `add_credentials("copernicus", client_id=..., client_secret=...)` |
| Service Account | google_earth_engine | `add_credentials("google_earth_engine", service_account=..., key_file=...)` |

**Next:** `03_satellite_data_search.ipynb` — advanced search techniques